# Fase 11 — **KONTROL**: k5-Aug dengan jalur deformable = duplikat row-forward (AdamW-auto)

**Pertanyaan.** Fase 10 (k5-Aug dgn offset yang DIPELAJARI) tidak melatih di AdamW: loss latih naik setelah epoch 3,
validasi kacau sejak epoch 1 dan NaN di epoch 19–20. B2-auto pada resep yang sama sukses (0.6796). Apakah penyebabnya
**urutan/resampling adaptif** pada jalur deformable, atau sesuatu pada arsitektur K=5 itu sendiri?

**Kontrol.** Arsitektur, parameter (6.215.245), resep, seed, dan data IDENTIK dgn Fase 10. Satu-satunya beda:
`DeformableScan.forward` diganti di runtime menjadi identitas — slot ke-5 = **duplikat PERSIS scan row-forward**
(tanpa offset, tanpa resampling, tanpa `argsort`). Tidak mengubah `deform_ss2d.py` (hash tetap 67544587…).
Parameter jaringan offset ada tapi TIDAK dipakai (tidak menerima gradien).

| Hasil kontrol | Artinya |
|---|---|
| **Melatih normal** (loss turun terus, val ≈ train, mAP naik) | urutan/resampling adaptif = pengganggu; mAP akhir menjawab 'K=5 aditif tanpa adaptivitas' vs B2-auto 0.6796 |
| **Ikut divergen** | penyebab ada di tempat lain (arsitektur K=5, ssm_ratio 1.7, dst.) — urutan token bukan tersangka utama |

⚠️ Kontrol ini **TIDAK menguji adaptivitas** — justru menghilangkannya. Ia memisahkan dua penjelasan.

**Titik keputusan awal (epoch ~20):** Fase 10 mencapai minimum train `cls_loss` 3.875 di epoch 3 lalu naik (5.79 di epoch 20) dan val
`cls_loss` 15–455 lalu NaN. Kalau kontrol menunjukkan pola yg sama, hentikan lebih awal.

**+ Pencatat norma gradien** (sel 7b) — hanya membaca; menguji dugaan 'gradien STE besar + clip_grad_norm=10'.

## 1. Mount Drive + Kredensial Kaggle

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, shutil, subprocess
DRIVE_ROOT   = "/content/drive/MyDrive/demamba-yolowd"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True); os.makedirs(f"{DRIVE_ROOT}/analysis", exist_ok=True)
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(f"{DRIVE_ROOT}/kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
subprocess.run([sys.executable,"-m","pip","install","-q","kaggle"], check=True)
print("Drive + Kaggle siap")

Mounted at /content/drive
Drive + Kaggle siap


## 2. Dataset + GPU

In [2]:
import torch
DATA_DIR = "/content/data"
if not glob.glob(f"{DATA_DIR}/**/*.txt", recursive=True):
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(["kaggle","datasets","download","-d","benyxu/dsf-yolo-steel-tube-dataset",
                    "-p",DATA_DIR,"--unzip"], check=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")
print(f"cpu_count={os.cpu_count()} -> workers={min(os.cpu_count()//max(torch.cuda.device_count(),1),8)}")
assert torch.cuda.is_available(), "GPU tidak aktif -- Runtime > Change runtime type"

GPU: NVIDIA A100-SXM4-40GB
cpu_count=12 -> workers=8


## 3. 🔧 Konfigurasi

In [3]:
SEED, TARGET_EPOCHS = 1, 150
OPTIMIZER   = "auto"          # sama dgn Fase 9 & 10. lr0 & momentum DIABAIKAN oleh "auto"
USE_COS_LR  = False
MODEL_LABEL = "K5dup-auto"    # label BEDA dari B2-auto / K5-auto -> baris CSV tidak bertabrakan
RUN_NAME    = f"k5dup_seed{SEED}_{TARGET_EPOCHS}ep_auto"
EXPECTED_PARAMS = 6_215_245   # identik dgn k5-Aug (jaringan offset tetap ada, hanya tak dipakai)
DEFORM_SRC  = f"{DRIVE_ROOT}/code/deform_ss2d.py"   # versi grid-fix; TIDAK diubah

print("varian   : k5-Aug KONTROL (slot ke-5 = duplikat row-forward; offset tak dipakai)")
print(f"RUN_NAME : {RUN_NAME}   label CSV: {MODEL_LABEL}")
print("resep    : optimizer=auto -> AdamW lr=0.000833 (identik Fase 9/10)")
print("rujukan  : B2-auto 0.6796 | K5-Aug offset dipelajari: DIVERGEN (epoch 20: train cls 5.79, val NaN)")

varian   : k5-Aug KONTROL (slot ke-5 = duplikat row-forward; offset tak dipakai)
RUN_NAME : k5dup_seed1_150ep_auto   label CSV: K5dup-auto
resep    : optimizer=auto -> AdamW lr=0.000833 (identik Fase 9/10)
rujukan  : B2-auto 0.6796 | K5-Aug offset dipelajari: DIVERGEN (epoch 20: train cls 5.79, val NaN)


## 4. Clone + Patch + Build Kernel

Urutan patch: **1.x** (SSM_CLASS) → **2.1** (impor) → **3.x** (registrasi P1) → **4.x/5.1**
(`torch.load`) → **K5-1..4** (`use_proj` + wrapper) → **K5-5** (registrasi K5) → **K5-6** (YAML).

`deform_ss2d.py` diambil dari Drive (**versi yang grid-nya sudah diperbaiki**), bukan dari repo.

In [4]:
import re, py_compile
REPO = "/content/Mamba-YOLO"
os.chdir("/content")
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","https://github.com/HZAI-ZJNU/Mamba-YOLO.git",REPO], check=True)

assert os.path.exists(DEFORM_SRC), f"letakkan deform_ss2d.py (versi grid diperbaiki) di {DEFORM_SRC}"
shutil.copy(DEFORM_SRC, f"{REPO}/ultralytics/nn/modules/deform_ss2d.py")
import hashlib
print("sha256 deform_ss2d.py:", hashlib.sha256(open(DEFORM_SRC,"rb").read()).hexdigest()[:16],
      "(versi perbaikan grid = 67544587...)")

def patch(path, subs):
    p = f"{REPO}/{path}"; src = open(p).read()
    for old, new, tag in subs:
        if new in src: print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new); print(f"  OK {tag}")
    open(p, "w").write(src); py_compile.compile(p, doraise=True)

M, D, T = "ultralytics/nn/modules/mamba_yolo.py", "ultralytics/nn/modules/deform_ss2d.py", "ultralytics/nn/tasks.py"
print("### 1.x registrasi SSM_CLASS")
patch(M, [("class XSSBlock(nn.Module):\n    def __init__(","class XSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(","1.1"),
 ("nn.Sequential(*(SS2D(d_model=self.hidden_dim,","nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,","1.2"),
 ("class VSSBlock(nn.Module):\n    def __init__(","class VSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(","1.3"),
 ("            self.op = SS2D(","            self.op = self.SSM_CLASS(","1.4")])
print("### 2.1 + K5-1..3 (use_proj)")
patch(D, [("from .mamba_yolo import SS2D","from .mamba_yolo import SS2D, VSSBlock, XSSBlock","2.1"),
 ('    def __init__(self, in_ch: int, kernel_size: int):\n        super().__init__()\n        pad = kernel_size // 2 if kernel_size > 1 else 0\n        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1)',
  '    def __init__(self, in_ch: int, kernel_size: int, use_proj: bool = True):\n        super().__init__()\n        pad = kernel_size // 2 if kernel_size > 1 else 0\n        self.use_proj = use_proj\n        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1) if use_proj else nn.Identity()', "K5-1"),
 ('    def __init__(self, embed_dim: int, kernel_size: int = 5):\n        super().__init__()\n        self.conv_offset = ConvOffset(embed_dim, kernel_size)',
  '    def __init__(self, embed_dim: int, kernel_size: int = 5, use_proj: bool = True):\n        super().__init__()\n        self.conv_offset = ConvOffset(embed_dim, kernel_size, use_proj=use_proj)', "K5-2"),
 ('                 deform_mode: str = "k3_replace",\n                 deform_stage: int = 0,\n                 deform_kernel_size: int = None,\n                 **kwargs):',
  '                 deform_mode: str = "k3_replace",\n                 deform_stage: int = 0,\n                 deform_kernel_size: int = None,\n                 deform_use_proj: bool = True,\n                 **kwargs):', "K5-3a"),
 ('DeformableScan(d_inner, kernel_size=ks) for _ in range(n_deform)',
  'DeformableScan(d_inner, kernel_size=ks, use_proj=deform_use_proj) for _ in range(n_deform)', "K5-3b")])

print("### K5-4 wrapper k5-Aug")
K5 = '''

class DeformVSSBlockK5(VSSBlock):
    """VSSBlock varian k5-Aug (ADITIF, K=4 -> K=5). ssm_ratio 1.7, tanpa ConvOffset.proj."""
    SSM_CLASS = partial(DeformSS2D, deform_mode="k5_augment",
                        deform_kernel_size=5, deform_use_proj=False)
    def __init__(self, *args, ssm_ratio=2.0, **kwargs):
        super().__init__(*args, ssm_ratio=1.7, **kwargs)


class DeformXSSBlockK5(XSSBlock):
    """XSSBlock varian k5-Aug - neck. Lihat DeformVSSBlockK5."""
    SSM_CLASS = partial(DeformSS2D, deform_mode="k5_augment",
                        deform_kernel_size=5, deform_use_proj=False)
    def __init__(self, *args, ssm_ratio=2.0, **kwargs):
        super().__init__(*args, ssm_ratio=1.7, **kwargs)
'''
_p = f"{REPO}/{D}"
if "DeformVSSBlockK5" not in open(_p).read():
    open(_p, "a").write(K5); print("  OK K5-4")
else: print("  ~ K5-4: skip")
py_compile.compile(_p, doraise=True)

print("### 3.x + K5-5 registrasi (P1 DAN K5 -- P1 wajib ikut agar regresi aman)")
patch(T, [("    XSSBlock\n)\n",
  "    XSSBlock\n)\nfrom ultralytics.nn.modules.deform_ss2d import (DeformVSSBlock, DeformXSSBlock,\n                                                DeformVSSBlockK5, DeformXSSBlockK5)\n","3.1+K5-5"),
 ("SimpleStem, VisionClueMerge, VSSBlock, XSSBlock\n        }:",
  "SimpleStem, VisionClueMerge, VSSBlock, XSSBlock, DeformVSSBlock, DeformXSSBlock, DeformVSSBlockK5, DeformXSSBlockK5\n        }:","3.2+K5-5b"),
 ("RepC3, XSSBlock}:","RepC3, XSSBlock, DeformXSSBlock, DeformXSSBlockK5}:","3.3+K5-5c"),
 ('            ckpt = torch.load(file, map_location="cpu")\n\n    except ModuleNotFoundError as e:',
  '            ckpt = torch.load(file, map_location="cpu", weights_only=False)\n\n    except ModuleNotFoundError as e:',"4.1"),
 ('        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu")',
  '        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu", weights_only=False)',"4.2")])
patch("ultralytics/utils/torch_utils.py", [
 ('    x = torch.load(f, map_location=torch.device("cpu"))\n    if "model" not in x:',
  '    x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)\n    if "model" not in x:',"5.1")])

print("### K5-6 YAML")
_y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
_y = re.sub(r"\bVSSBlock\b","DeformVSSBlockK5",_y); _y = re.sub(r"\bXSSBlock\b","DeformXSSBlockK5",_y)
K5_YAML = f"{REPO}/ultralytics/cfg/models/mamba-yolo/mamba-yolo-T-k5aug.yaml"
open(K5_YAML,"w").write(_y); print("  OK", os.path.basename(K5_YAML))

# ---- kernel selective_scan (cache di Drive) ----
subprocess.run([sys.executable,"-m","pip","install","-q","ninja"], check=True)
os.environ["MAX_JOBS"] = str(max(1,int(subprocess.run(["nproc"],capture_output=True,text=True).stdout.strip() or "2")))
cap = torch.cuda.get_device_capability(0)
SS_TAG = f"sm{cap[0]}{cap[1]}_torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
SS_DIR = f"{DRIVE_ROOT}/wheels/selective_scan_{SS_TAG}"; os.makedirs(SS_DIR, exist_ok=True)
whl = glob.glob(f"{SS_DIR}/*.whl")
if whl:
    print(f"Wheel cache ({SS_TAG})"); subprocess.run([sys.executable,"-m","pip","install"]+whl, check=True)
else:
    print(f"Compile kernel sekali ({SS_TAG})")
    subprocess.run([sys.executable,"-m","pip","wheel",".","--no-deps","-v","-w",SS_DIR], cwd=f"{REPO}/selective_scan", check=True)
    subprocess.run([sys.executable,"-m","pip","install"]+glob.glob(f"{SS_DIR}/*.whl"), check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e","."], cwd=REPO, check=True)
for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]: del sys.modules[k]
import selective_scan_cuda_core; print("selective_scan_cuda_core OK")
subprocess.run([sys.executable,"-m","pip","install","-q","seaborn","thop","timm","einops"], check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","wandb","ray","tensorboard","tensorboardX",
                "tensorboard-data-server","clearml","comet_ml","dvclive","mlflow"], check=False)
print("Lingkungan siap")

sha256 deform_ss2d.py: 675445871a2e4d55 (versi perbaikan grid = 67544587...)
### 1.x registrasi SSM_CLASS
  OK 1.1
  OK 1.2
  OK 1.3
  OK 1.4
### 2.1 + K5-1..3 (use_proj)
  ~ 2.1: skip
  OK K5-1
  OK K5-2
  OK K5-3a
  OK K5-3b
### K5-4 wrapper k5-Aug
  OK K5-4
### 3.x + K5-5 registrasi (P1 DAN K5 -- P1 wajib ikut agar regresi aman)
  OK 3.1+K5-5
  OK 3.2+K5-5b
  OK 3.3+K5-5c
  OK 4.1
  OK 4.2
  OK 5.1
### K5-6 YAML
  OK mamba-yolo-T-k5aug.yaml
Wheel cache (sm80_torch2.11.0_py313)
selective_scan_cuda_core OK
Lingkungan siap


## 5. data.yaml — identik dgn 40 run DSF sebelumnya

In [5]:
import yaml
from pathlib import Path
cands = [p.parent.parent for p in Path(DATA_DIR).rglob("images/train2021") if p.is_dir()]
assert cands, f"struktur images/train2021 tidak ditemukan di {DATA_DIR}"
DSF_BASE = str(cands[0].resolve())
NAMES = ["air-hole","bite-edge","broken-arc","crack","dense-gas-hole","non-fusion","slag-inclusion","unfused-hole"]
DATA_YAML = f"{REPO}/ultralytics/cfg/datasets/dsf-yolo.yaml"
yaml.safe_dump({"path":DSF_BASE,"train":"images/train2021","val":"images/val2021",
                "nc":len(NAMES),"names":NAMES}, open(DATA_YAML,"w"), sort_keys=False)
for s in ["train2021","val2021"]:
    d=Path(DSF_BASE)/"images"/s
    print(f"  {s}: {len([f for f in d.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}])} citra")
assert (Path(DSF_BASE)/"labels"/"train2021").is_dir(), "folder labels/ tidak ada"
print(f"\nDSF_BASE: {DSF_BASE}")

  train2021: 2726 citra
  val2021: 682 citra

DSF_BASE: /content/data/steel-tube-dataset-all/yolo


## 6. Pre-flight WAJIB — **hentikan kalau angkanya meleset**

Tiga hal dicek: k5-Aug = 6.215.245 & K=5 & 10 modul, **dan** regresi P1/B2 lama tetap utuh.

In [6]:
os.chdir(REPO)
from copy import deepcopy
from ultralytics import YOLO
from ultralytics.nn.tasks import parse_model, yaml_model_load
from ultralytics.nn.modules.deform_ss2d import DeformSS2D

d = yaml_model_load(K5_YAML); d["nc"] = len(NAMES)
m,_ = parse_model(deepcopy(d), ch=3, verbose=False)
n_par = sum(p.numel() for p in m.parameters())
ds = [x for x in m.modules() if isinstance(x, DeformSS2D)]
print(f"k5-Aug : {n_par:,} parameter (harap {EXPECTED_PARAMS:,})   Δ vs B2 = {(n_par-5_986_912)/5_986_912*100:+.2f}%")
print(f"         {len(ds)} modul DeformSS2D (harap 10), K = {sorted({x.K for x in ds})} (harap [5])")
assert n_par == EXPECTED_PARAMS, "PARAMETER MELESET -- JANGAN mulai training"
assert len(ds) == 10 and {x.K for x in ds} == {5}, "mode/jumlah modul salah"
del m, ds

# regresi: P1 & B2 lama harus tetap utuh
_y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
_p1 = re.sub(r"\bXSSBlock\b","DeformXSSBlock", re.sub(r"\bVSSBlock\b","DeformVSSBlock",_y))
open("/content/p1_regresi.yaml","w").write(_p1)
for lbl,f,exp in [("P1 lama","/content/p1_regresi.yaml",5_930_869),
                  ("B2 lama",f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml",5_986_912)]:
    dd = yaml_model_load(f); dd["nc"] = len(NAMES)
    mm,_ = parse_model(deepcopy(dd), ch=3, verbose=False)
    pp = sum(p.numel() for p in mm.parameters()); del mm
    print(f"  regresi {lbl}: {pp:,} (harap {exp:,}) -> {'AMAN' if pp==exp else 'RUSAK'}")
    assert pp == exp, f"patch merusak {lbl} -- hentikan"
print("\nPre-flight LOLOS")

WARNING ⚠️ no model scale passed. Assuming scale='T'.


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:103: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:130: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd


k5-Aug : 6,215,245 parameter (harap 6,215,245)   Δ vs B2 = +3.81%
         10 modul DeformSS2D (harap 10), K = [5] (harap [5])
WARNING ⚠️ no model scale passed. Assuming scale='T'.


/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(
/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:277: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.ss2d = nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,


  regresi P1 lama: 5,930,869 (harap 5,930,869) -> AMAN
WARNING ⚠️ no model scale passed. Assuming scale='T'.
  regresi B2 lama: 5,986,912 (harap 5,986,912) -> AMAN

Pre-flight LOLOS


## 6b. KONTROL — jalur deformable = duplikat row-forward

`DeformableScan.forward` diganti di runtime. Dijalankan **setelah** pre-flight (yg sudah memastikan 6.215.245 parameter, K=5,
10 modul, dan regresi P1/B2) dan **sebelum** training. Verifikasi fungsional: slot ke-5 harus sama persis dgn slot row-forward, dan
`_merge_ys` harus kebalikan `_build_xs`.

In [7]:
import torch
import ultralytics.nn.modules.deform_ss2d as _D

def _identity_forward(self, x):
    """KONTROL: jalur 'deformable' = duplikat PERSIS scan row-forward
    (tanpa offset, tanpa resampling spasial, tanpa argsort)."""
    B, C, H, W = x.shape
    idx = torch.arange(H * W, device=x.device).unsqueeze(0).expand(B, -1)
    return x.flatten(2), idx

_D.DeformableScan.forward = _identity_forward
print("KONTROL aktif: DeformableScan.forward -> identitas (duplikat row-forward)")

# verifikasi fungsional (PyTorch murni; tidak butuh kernel CUDA)
_m = _D.DeformSS2D(d_model=16, ssm_ratio=1.0, deform_mode="k5_augment", deform_use_proj=False)
_x = torch.randn(2, 16, 10, 12)
with torch.no_grad():
    _xs, _idx = _m._build_xs(_x)
    assert _xs.shape[1] == 5, "K harus 5"
    assert torch.equal(_xs[:, 4], _xs[:, 0]), "slot ke-5 BUKAN duplikat row-forward"
    _y = _m._merge_ys(_xs, _idx, 10, 12)
    assert torch.allclose(_y, 5 * _x.flatten(2), atol=1e-5), "merge bukan kebalikan scan"
print("Verifikasi kontrol OK: slot-5 == slot-0 (row-forward) dan merge = kebalikan scan")
del _m, _x, _xs, _idx, _y


KONTROL aktif: DeformableScan.forward -> identitas (duplikat row-forward)
Verifikasi kontrol OK: slot-5 == slot-0 (row-forward) dan merge = kebalikan scan


## 7. Status checkpoint

In [8]:
last_ckpt = Path(DRIVE_OUTPUT)/RUN_NAME/"weights"/"last.pt"
if not last_ckpt.exists():
    STATUS="BARU"; print("STATUS: BARU")
else:
    raw = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
    ep, ho = raw.get("epoch",-1), raw.get("optimizer") is not None
    STATUS = "BELUM_SELESAI" if (ho and 0 <= ep < TARGET_EPOCHS-1) else "SELESAI"
    print(f"STATUS: {STATUS} (epoch={ep}, optimizer={ho})")

STATUS: BARU


## 7b. Pencatat norma gradien (HANYA MEMBACA)

Mencatat per epoch norma gradien **global sebelum clip** (median, p95, maks, % langkah yg di-clip >10, % non-finite) ke
`grad_norm.csv`. Latar belakang: Ultralytics memakai `clip_grad_norm_(max_norm=10)`; kalau gradien STE pada jalur deformable
sangat besar, clip menekan SELURUH gradien jaringan. Di kontrol tanpa STE, norma ini seharusnya wajar — jadi ia menjadi
**pembanding dasar** utk run offset-dipelajari berikutnya.

Diuji pada training Ultralytics sungguhan (CPU): loss dgn vs tanpa pencatat **identik persis; pembungkus mengembalikan
nilai asli; tidak ada `.item()` per langkah.

In [9]:
import csv
import torch
from pathlib import Path

class GradNormLogger:
    """Mencatat norma gradien GLOBAL (sebelum clip) tiap langkah optimizer.
    HANYA MEMBACA: pembungkus mengembalikan nilai asli; tidak ada .item() per langkah."""
    def __init__(self):
        self.norms, self._orig = [], None

    def attach(self):
        if self._orig is not None:
            return                                             # idempoten (juga saat resume)
        self._orig = torch.nn.utils.clip_grad_norm_
        def wrapped(params, max_norm, *a, **k):
            n = self._orig(params, max_norm, *a, **k)          # nilai kembalian = norma SEBELUM clip
            self.norms.append(n.detach().float())
            return n
        torch.nn.utils.clip_grad_norm_ = wrapped
        print("[GradNormLogger] terpasang (clip_grad_norm_ dibungkus; perilaku tidak berubah)")

    def flush(self, epoch, run_dir):
        if not self.norms:
            return
        v = torch.stack(self.norms); fin = v[torch.isfinite(v)]
        row = dict(epoch=epoch, n_steps=len(v),
                   frac_nonfinite=(~torch.isfinite(v)).float().mean().item(),
                   mean=fin.mean().item() if len(fin) else float("nan"),
                   median=fin.median().item() if len(fin) else float("nan"),
                   p95=torch.quantile(fin, 0.95).item() if len(fin) else float("nan"),
                   max=fin.max().item() if len(fin) else float("nan"),
                   frac_clipped=(fin > 10.0).float().mean().item() if len(fin) else float("nan"))
        path = Path(run_dir) / "grad_norm.csv"; new = not path.exists()
        with open(path, "a", newline="") as f:
            wr = csv.DictWriter(f, fieldnames=list(row.keys()))
            if new: wr.writeheader()
            wr.writerow(row)
        if epoch == 1 or epoch % 10 == 0:
            print(f"[GradNormLogger] epoch {epoch}: norma gradien median {row['median']:.2f} | p95 {row['p95']:.2f} | "
                  f"maks {row['max']:.1f} | di-clip (>10): {row['frac_clipped']:.1%} | non-finite: {row['frac_nonfinite']:.1%}")
        self.norms = []


## 8. Training

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

monitor = GradNormLogger()
def _attach(tr): monitor.attach()                                  # dipasang di awal training (juga saat resume)
def _flush(tr):  monitor.flush(tr.epoch + 1, tr.save_dir)       # tulis statistik norma gradien tiap epoch

if STATUS == "BARU":
    model = YOLO(K5_YAML)
    model.add_callback("on_train_start", _attach); model.add_callback("on_train_epoch_end", _flush)
    model.train(data=DATA_YAML, epochs=TARGET_EPOCHS,
        batch=8, imgsz=640, device=0, amp=True, seed=SEED,
        patience=0, cache=True, deterministic=True, workers=8,
        optimizer=OPTIMIZER, cos_lr=USE_COS_LR,
        project=DRIVE_OUTPUT, name=RUN_NAME, plots=False)
elif STATUS == "BELUM_SELESAI":
    model = YOLO(str(last_ckpt))
    model.add_callback("on_train_start", _attach); model.add_callback("on_train_epoch_end", _flush)
    model.train(resume=True)
else:
    model = YOLO(str(last_ckpt)); print("Sudah selesai -- tidak training ulang.")

SAVE_DIR = model.trainer.save_dir if getattr(model,"trainer",None) else Path(DRIVE_OUTPUT)/RUN_NAME
print(f"\nSAVE_DIR: {SAVE_DIR}")
print("CATAT baris 'optimizer:' di log (AdamW + lr yang dipilih framework).")
print("CATAT: apakah cls_loss TERUS TURUN (Fase 10: min di epoch 3 lalu naik)? val_cls ~ train_cls? val NaN?")
fin = torch.load(str(Path(SAVE_DIR)/"weights"/"last.pt"), map_location="cpu", weights_only=False)
print("final_eval selesai" if fin.get("optimizer") is None else f"BELUM tuntas (epoch={fin.get('epoch')}) -- jalankan ulang utk resume")

_chk = [(n, float(m.conv_offset.out.weight.abs().sum())) for n, m in model.model.named_modules() if hasattr(m, "conv_offset")]
print(f"KONTROL valid -- bobot 'out' jaringan offset tetap 0 di semua blok: {all(v == 0.0 for _, v in _chk)} ({len(_chk)} blok)")

WARNING ⚠️ no model scale passed. Assuming scale='T'.
New https://pypi.org/project/ultralytics/8.4.166 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=/content/Mamba-YOLO/ultralytics/cfg/models/mamba-yolo/mamba-yolo-T-k5aug.yaml, data=/content/Mamba-YOLO/ultralytics/cfg/datasets/dsf-yolo.yaml, epochs=150, time=None, patience=0, batch=8, imgsz=640, save=True, save_period=-1, cache=True, device=0, workers=8, project=/content/drive/MyDrive/demamba-yolowd/output, name=k5dup_seed1_150ep_auto, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=1, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou

100%|██████████| 755k/755k [00:00<00:00, 19.7MB/s]

Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      5136  ultralytics.nn.modules.mamba_yolo.SimpleStem [3, 32, 3]                    
  1                  -1  1     36516  ultralytics.nn.modules.deform_ss2d.DeformVSSBlockK5[32, 32]                      
  2                  -1  1      8384  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[32, 64]                      
  3                  -1  1    111046  ultralytics.nn.modules.deform_ss2d.DeformVSSBlockK5[64, 64]                      
  4                  -1  1     33152  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[64, 128]                     
  5                  -1  3   1124949  ultralytics.nn.modules.deform_ss2d.DeformVSSBlockK5[128, 128]                    
  6                  -1  1    131840  ultralytics.nn.modules.mamba_yolo.VisionClueM


/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(
/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:277: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.ss2d = nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,


mamba-YOLO-T-k5aug summary: 544 layers, 6215245 parameters, 6215229 gradients, 13.1 GFLOPs

Freezing layer 'model.21.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...


...

                   all        682       1381      0.952      0.945      0.967      0.666



150 epochs completed in 3.644 hours.
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/k5dup_seed1_150ep_auto/weights/last.pt, 12.8MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/k5dup_seed1_150ep_auto/weights/best.pt, 12.8MB

Validating /content/drive/MyDrive/demamba-yolowd/output/k5dup_seed1_150ep_auto/weights/best.pt...
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
mamba-YOLO-T-k5aug summary: 528 layers, 6213901 parameters, 0 gradients, 13.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:04<00:00,  9.32it/s]

                   all        682       1381      0.952       0.95      0.967      0.668
              air-hole        406       1070      0.855      0.834      0.904      0.423
             bite-edge          8          8      0.934          1      0.995      0.693
            broken-arc         74         74      0.947      0.967      0.989      0.856
                 crack         23         23      0.985      0.826      0.891      0.525
        dense-gas-hole         43         43      0.946      0.977      0.983       0.78
            non-fusion         40         41          1      0.994      0.995      0.718
        slag-inclusion         27         27      0.983          1      0.995      0.641


          unfused-hole         95         95      0.969      0.999      0.983      0.706
Speed: 0.1ms preprocess, 4.1ms inference, 0.0ms loss, 0.8ms postprocess per image

SAVE_DIR: /content/drive/MyDrive/demamba-yolowd/output/k5dup_seed1_150ep_auto
CATAT baris 'optimizer:' di log (AdamW + lr yang dipilih framework).
CATAT: apakah cls_loss TERUS TURUN (Fase 10: min di epoch 3 lalu naik)? val_cls ~ train_cls? val NaN?
final_eval selesai
KONTROL valid -- bobot 'out' jaringan offset tetap 0 di semua blok: True (10 blok)


## 9. Evaluasi + CSV (bergabung dgn hasil resep auto lainnya)

In [ ]:
import pandas as pd
OUT_CSV = f"{DRIVE_ROOT}/analysis/per_class_results_recipe.csv"
best = Path(SAVE_DIR)/"weights"/"best.pt"
assert best.exists(), "best.pt belum ada"
r = YOLO(str(best)).val(data=DATA_YAML, split="val", verbose=False)
base = dict(model=MODEL_LABEL, seed=SEED, ckpt_path=str(best))
rows = [{**base,"class":"ALL","mAP50_95":r.box.map,"mAP50":r.box.map50,
         "precision":r.box.mp,"recall":r.box.mr}]
for i,c in enumerate(NAMES):
    rows.append({**base,"class":c,"mAP50_95":float(r.box.maps[i]),
                 "mAP50":float("nan"),"precision":float("nan"),"recall":float("nan")})
df = pd.DataFrame(rows)
if os.path.exists(OUT_CSV):
    df = pd.concat([pd.read_csv(OUT_CSV), df]).drop_duplicates(["model","seed","class"], keep="last")
df.to_csv(OUT_CSV, index=False); shutil.copy(OUT_CSV, "/content/per_class_results_recipe.csv")
v = df[(df["class"]=="ALL") & (df.model==MODEL_LABEL)].mAP50_95.iloc[0]
print(df[df["class"]=="ALL"][["model","seed","mAP50_95","mAP50","precision","recall"]].to_string(index=False))
print(f"\n  B2-auto (baseline yang benar) : 0.6796")
print(f"  K5dup (RUN INI, kontrol)      : {v:.4f}   ({v-0.6796:+.4f})")
print(f"  K5-Aug offset dipelajari      : DIVERGEN (dihentikan @epoch 20)")
print(f"\n  Kelas elongated (crack, broken-arc) -- fokus hipotesis:")
for c in ["crack","broken-arc"]:
    print(f"    {c:12s} {df[(df.model==MODEL_LABEL)&(df['class']==c)].mAP50_95.iloc[0]:.4f}")
for _src,_dst in [(Path(SAVE_DIR)/"grad_norm.csv","/content/k5dup_grad_norm.csv"),
                  (Path(SAVE_DIR)/"results.csv","/content/k5dup_results.csv")]:
    if _src.exists(): shutil.copy(_src,_dst); print("salinan:",_dst)
print(f"\nTersimpan: {OUT_CSV}")
print("KIRIM ke sesi analisis: per_class_results_recipe.csv, k5dup_grad_norm.csv, k5dup_results.csv")

Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
mamba-YOLO-T-k5aug summary: 528 layers, 6213901 parameters, 0 gradients, 13.1 GFLOPs


val: Scanning /content/data/steel-tube-dataset-all/yolo/labels/val2021.cache... 682 images, 2 backgrounds, 0 corrupt: 100%|██████████| 682/682 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:07<00:00,  5.94it/s]


                   all        682       1381      0.946       0.95      0.966      0.668
Speed: 0.1ms preprocess, 6.1ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/Mamba-YOLO/runs/detect/val
       model  seed  mAP50_95    mAP50  precision   recall
     B2-auto     1  0.679606 0.970531   0.951208 0.947711
YOLOv8n-auto     1  0.663947 0.967367   0.941636 0.936849
YOLOv8s-auto     1  0.697972 0.979487   0.959881 0.933910
  K5dup-auto     1  0.668221 0.966149   0.946420 0.949627

  B2-auto (baseline yang benar) : 0.6796
  K5dup (RUN INI, kontrol)      : 0.6682   (-0.0114)
  K5-Aug offset dipelajari      : DIVERGEN (dihentikan @epoch 20)

  Kelas elongated (crack, broken-arc) -- fokus hipotesis:
    crack        0.5220
    broken-arc   0.8610
salinan: /content/k5dup_grad_norm.csv
salinan: /content/k5dup_results.csv

Tersimpan: /content/drive/MyDrive/demamba-yolowd/analysis/per_class_results_recipe.csv
KIRIM ke sesi analisis: per_class_results_recipe.csv, k5

## 10. Matikan Runtime (Jalankan Terakhir)

In [ ]:
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")